In [160]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests

In [161]:
def probable_role(df):

    roles = {}
    date_pattern = r"^(\d{4}[-/.]\d{1,2}[-/.]\d{1,2}|\d{1,2}[-/.]\d{1,2}[-/.]\d{4})(\s+\d{1,2}:\d{2}(:\d{2})?)?$"

    for col in df.columns:

        if (df[col] == 0).all() or df[col].isna().all():
            roles[col] = "Unknown or mixed type"
            continue

        if df[col].dropna().dtype == 'bool':
            roles[col] = "Boolean"
            continue

        if len(df[col].dropna().unique()) == 2:
                roles[col] = "Boolean"
                continue

        if pd.api.types.is_numeric_dtype(df[col].dropna()):
            roles[col] = "Numeric"
            continue

        if pd.api.types.is_datetime64_any_dtype(df[col].dropna()):
            roles[col] = "Date-like"
            continue

        if df[col].dropna().astype(str).str.match(date_pattern).mean() >= 0.8:
            roles[col] = "Date-like"
            continue

        if len(df[col].dropna().unique())/len(df[col].dropna()) > 0.90:
            roles[col] = "Identifier-like"
            continue

        if (len(df[col].dropna().unique()) > 2) & (len(df[col].dropna().unique())/len(df[col].dropna()) < 0.05):
            roles[col] = "Categorical"
            continue

        if pd.api.types.is_object_dtype(df[col].dropna()) or pd.api.types.is_string_dtype(df[col].dropna()):
            roles[col] = "Free-text field"
            continue
            
        else:
            roles[col] = "Unknown or mixed type"
   
    return roles


In [162]:
def get_dataset_overview(df, dataset_name, roles):
    overview = {
        'filename': dataset_name,
        'rows': len(df),
        'columns': len(df.columns),
        'column_name': {},
        'technical_type': {},
        'non_missing_count': {},
        'missing_percent': {},
        'unique_values': {},
        'probable_role': {},
    }
    for col in df.columns:
        missing_count = int(df[col].isna().sum())
        overview['column_name'] = col
        overview['technical_type'][col] = str(df[col].dtype)
        overview['non_missing_count'][col] = len(df[col]) - missing_count
        overview['missing_percent'][col] = round(missing_count / len(df[col]), 4)
        overview['unique_values'][col] = int(df[col].dropna().nunique())
        overview['probable_role'][col] = roles[col]
    return overview

In [163]:
def get_data_quality_profile(df, roles):
    
    missing_threshold = 0.30
    sensitive_keywords = ['ssn', 'password', 'credit', 'phone', 'email',
                           'address', 'zip', 'dob', 'birth', 'salary', 'income', 'social_security']

    profile = {
        'Number of duplicate rows in each columns': {},
        'Columns containing only one distinct value': [],
        'Columns with a high percentage of missing values': [],
        'Columns that have mixed or inconsistent data type': [col for col, role in roles.items() if role == "Unknown or mixed type"],
        'Columns that have identifier-like or extremely high-cardinality data type': [col for col, role in roles.items() if role == "Identifier-like"],
        'Columns that might contain sensitive data based on column name': []
    }

    for col in df.columns:
        profile['Number of duplicate rows in each columns'][col] = int(df.dropna().duplicated().sum())
        if df[col].dropna().nunique() == 1:
            profile['Columns containing only one distinct value'].append(col)
        if round(int(df[col].isna().sum()) / len(df[col]), 4) >= missing_threshold:
            profile['Columns with a high percentage of missing values'].append(col)
        if any(keyword in col.lower() for keyword in sensitive_keywords):
            profile['Columns that might contain sensitive data based on column name'].append(col)
            
    return profile

In [164]:
def descriptive_statistics(df, roles):

    Categorical_Columns = [col for col, role in roles.items() if role == "Categorical"]
    Categorical_Columns_Stats = {
        'Number of unique categories in the column': {},
        'Most frequent category or categories': {},
        'Frequency count of top 10 categories': {},
        'Frequency count percentage of top 10 categories': {},
        'Top categories when the column contains many unique values': {}
    }

    Numeric_Columns = [col for col, role in roles.items() if role == "Numeric"]
    Numeric_Columns_Stats = {
        'Valid count of the column': {},
        'Missing count and percentage of the column': {},
        'Minimum value in the column': {},
        'Maximum value in the column': {},
        'Mean of the column': {},
        'Median of the column': {},
        'Mode of the column': {},
        'Standard deviation of the column': {},
        'First quartile of the column': {},
        'Third quartile of the column': {},
        'Interquartile of the column': {},
        'Outlier count and percentage of the column': {}
    }

    Stats = {
        'Categorical Columns descriptive statistics (if blank then no categorical columns found)': Categorical_Columns_Stats, 
        'Numerical Columns descriptive statistics (if blank then no numeric columns found)': Numeric_Columns_Stats
    }

    for col in Categorical_Columns:
        Categorical_Columns_Stats['Number of unique categories in the column'][col] = int(df[col].nunique())
        Categorical_Columns_Stats['Most frequent category or categories'][col] = df[col].value_counts().head(3)

        Categorical_Columns_Stats['Frequency count of top 10 categories'][col] = {}
        Categorical_Columns_Stats['Frequency count percentage of top 10 categories'][col] = {}

        for category, count in df[col].value_counts().head(10).items():
            Categorical_Columns_Stats['Frequency count of top 10 categories'][col][category] = int(count)
            Categorical_Columns_Stats['Frequency count percentage of top 10 categories'][col][category] = round((count / len(df[col]))*100,1)
            
        if df[col].nunique() > 10:
            Categorical_Columns_Stats['Top categories when the column contains many unique values'][col] = df[col].value_counts().head(3)

    for col in Numeric_Columns:
        valid_data = df[col].dropna()
        Numeric_Columns_Stats['Valid count of the column'][col] = len(valid_data)
        Numeric_Columns_Stats['Missing count and percentage of the column'][col] = {len(df[col].isna()), round((len(df[col].isna())/len(df[col]))*100,1)}
        Numeric_Columns_Stats['Minimum value in the column'][col] = float(valid_data.min())
        Numeric_Columns_Stats['Maximum value in the column'][col] = float(valid_data.max())
        Numeric_Columns_Stats['Mean of the column'][col] = float(valid_data.mean())
        Numeric_Columns_Stats['Median of the column'][col] = float(valid_data.median())
        Numeric_Columns_Stats['Mode of the column'][col] = float(valid_data.mode().iloc[0])
        Numeric_Columns_Stats['Standard deviation of the column'][col] = float(valid_data.std())
        q1 = valid_data.quantile(0.25)
        q3 = valid_data.quantile(0.75)
        iqr = q3 - q1
        outliers = valid_data[(valid_data < (q1 - 1.5 * iqr)) | (valid_data > (q3 + 1.5 * iqr))]
        Numeric_Columns_Stats['First quartile of the column'][col] = float(q1)
        Numeric_Columns_Stats['Third quartile of the column'][col] = float(q3)
        Numeric_Columns_Stats['Interquartile of the column'][col] = float(iqr)
        Numeric_Columns_Stats['Outlier count and percentage of the column'][col] = {int(len(outliers)), round((int(len(outliers))/len(df[col]))*100,1)}

    return Stats



In [165]:
def get_relationships_between_variables(df, roles):
    relationships = {}
    
    numeric_cols = [col for col, role in roles.items() if role == "Numeric"]
    
    if len(numeric_cols) > 1:
        corr_matrix = df[numeric_cols].corr()
        
        corr_unstacked = corr_matrix.unstack()
        corr_unstacked = corr_unstacked[corr_unstacked.abs() < 1.0].dropna()
        
        if not corr_unstacked.empty:
            relationships['correlation_matrix'] = corr_matrix
            
            positive_corr = corr_unstacked[corr_unstacked > 0]
            if not positive_corr.empty:
                strongest_pos = positive_corr.idxmax()
                relationships['strongest_positive_pair'] = strongest_pos
                relationships['strongest_positive_value'] = round(corr_matrix.loc[strongest_pos[0], strongest_pos[1]], 4)
            else:
                relationships['strongest_positive_pair'] = None
                relationships['strongest_positive_value'] = None
                
            negative_corr = corr_unstacked[corr_unstacked < 0]
            if not negative_corr.empty:
                strongest_neg = negative_corr.idxmin()
                relationships['strongest_negative_pair'] = strongest_neg
                relationships['strongest_negative_value'] = round(corr_matrix.loc[strongest_neg[0], strongest_neg[1]], 4)
            else:
                relationships['strongest_negative_pair'] = None
                relationships['strongest_negative_value'] = None
                
            relationships['skipped'] = False
        else:
            relationships['skipped'] = True
            relationships['reason'] = "Not enough variation or valid data points for correlation analysis."
    else:
        relationships['skipped'] = True
        relationships['reason'] = "Not enough numeric columns for correlation analysis."
        
    return relationships

In [166]:
def generate_adaptive_visualizations(df, roles, plots_dir):
    """Generates up to 5-10 dataset-driven plots based on inferred attribute roles."""
    generated_plots = []
    skipped_plots = {}
    sns.set_theme(style="whitegrid")
    
    os.makedirs(plots_dir, exist_ok=True)
    plot_num = 1

    numeric_cols = [col for col, role in roles.items() if role == 'Numeric']
    cat_cols = [col for col, role in roles.items() if role in ['Categorical', 'Boolean']]

    missing_data = df.isna().sum()
    if missing_data.sum() > 0:
        plt.figure(figsize=(10, 6))
        missing_data[missing_data > 0].plot(kind='bar', color='salmon')
        plt.title('Missing Values per Column')
        plt.xlabel('Columns')
        plt.ylabel('Count of Missing Values')
        plt.tight_layout()
        filename = f'plot_{plot_num:02d}_missing_values.png'
        plt.savefig(os.path.join(plots_dir, filename))
        plt.close()
        generated_plots.append(filename)
        plot_num += 1
    else:
        skipped_plots['Missing-value bar chart'] = "No missing values present in the dataset."

    if len(numeric_cols) >= 2:
        plt.figure(figsize=(10, 8))
        sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f")
        plt.title('Numeric Correlation Heatmap')
        plt.xlabel('Numeric Variables')
        plt.ylabel('Numeric Variables')
        plt.tight_layout()
        filename = f'plot_{plot_num:02d}_correlation.png'
        plt.savefig(os.path.join(plots_dir, filename))
        plt.close()
        generated_plots.append(filename)
        plot_num += 1
    else:
        skipped_plots['Correlation heatmap'] = "Fewer than 2 numeric columns available."

    hist_count = 0
    for col in numeric_cols[:3]: 
        plt.figure(figsize=(10, 6))
        sns.histplot(df[col].dropna(), kde=True, bins=30)
        plt.title(f'Distribution of {col}')
        plt.xlabel(col)
        plt.ylabel('Frequency / Density')
        plt.tight_layout()
        filename = f'plot_{plot_num:02d}_dist_{col[:10]}.png'
        plt.savefig(os.path.join(plots_dir, filename))
        plt.close()
        generated_plots.append(filename)
        plot_num += 1
        hist_count += 1

    if hist_count == 0 and len(numeric_cols) == 0:
        skipped_plots['Histogram'] = "No numeric columns available for distribution analysis."

    cat_count = 0
    for col in cat_cols[:3]:
        plt.figure(figsize=(10, 6))
        df[col].value_counts().head(10).plot(kind='bar', color='skyblue')
        plt.title(f'Top Categories in {col}')
        plt.xlabel('Categories')
        plt.ylabel('Frequency')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        filename = f'plot_{plot_num:02d}_cat_{col[:10]}.png'
        plt.savefig(os.path.join(plots_dir, filename))
        plt.close()
        generated_plots.append(filename)
        plot_num += 1
        cat_count += 1

    if cat_count == 0 and len(cat_cols) == 0:
        skipped_plots['Categorical frequency bar chart'] = "No categorical columns available."

    if len(numeric_cols) >= 2:
        plt.figure(figsize=(10, 6))
        sns.scatterplot(data=df, x=numeric_cols[0], y=numeric_cols[1], alpha=0.7)
        plt.title(f'Relationship between {numeric_cols[0]} and {numeric_cols[1]}')
        plt.xlabel(numeric_cols[0])
        plt.ylabel(numeric_cols[1])
        plt.tight_layout()
        filename = f'plot_{plot_num:02d}_scatterplot.png'
        plt.savefig(os.path.join(plots_dir, filename))
        plt.close()
        generated_plots.append(filename)
        plot_num += 1

    return {
        'generated_plots': generated_plots,
        'skipped_plots': skipped_plots,
        'total_generated': len(generated_plots)
    }

In [167]:
def run_llm_insights(summary, output_dir, use_llm=True, model_name="qwen2.5:0.5b"):
    prompt = f"""You are an expert data analyst at Google. 
I have profiled a dataset and generated a file in JSON format that includes dataset overview, data quality profile, stats and relationships between variables. 

For each of the following write one full sentence:
1. Give one data-quality finding and answer why this is important.
2. Give one distribution finding and answer why this is important.
3. Give an insightful comment on one categorical finding (if categorical columns exist) and answer why this is important.
4. Give an insightful comment on one relationship finding (if numeric columns exist) and answer why this is important.
5. State one limitation or potential-bias warning you find and answer why this is important.
6. Provide one recommended question for further investigation.

Data Summary:
{json.dumps(summary, indent=2)}

Do not calculate new statistics. Do not invent facts or units. Write clearly and professionally.
"""
    
    prompt_file = os.path.join(output_dir, 'llm_prompt.txt')
    with open(prompt_file, 'w', encoding='utf-8') as f:
        f.write(prompt)

    response_file = os.path.join(output_dir, 'llm_response.txt')

    if not use_llm:
        msg = "AI-generated narrative insights were skipped (use_llm=False)."
        with open(response_file, 'w', encoding='utf-8') as f:
            f.write(msg)
        return msg

    try:
        url = "http://localhost:11434/api/generate"
        payload = {
            "model": model_name,
            "prompt": prompt,
            "stream": False
        }
        
        response = requests.post(url, json=payload, timeout=60)
        response.raise_for_status()
        
        insights = response.json().get('response', 'No response text returned from LLM.')
        with open(response_file, 'w', encoding='utf-8') as f:
            f.write(insights)
            
        return insights

    except Exception as e:
        msg = f"AI-generated narrative insights were skipped because the local LLM model was unavailable.\nError details: {e}"
        with open(response_file, 'w', encoding='utf-8') as f:
            f.write(msg)
        return msg

In [168]:
def generate_markdown_report(dataset_name, summary, viz_info, llm_insights, output_dir):
    """Compiles findings and plots into the required report.md format."""
    report_path = os.path.join(output_dir, 'report.md')
    overview = summary['Dataset Overview']
    quality = summary['Data Quality']
    
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write(f"# Data Profiling & Evidence Report: {dataset_name}\n\n")
        
        f.write("## 1. Dataset Overview\n")
        f.write(f"- **Filename:** `{overview['filename']}`\n")
        f.write(f"- **Total Rows:** {overview['rows']}\n")
        f.write(f"- **Total Columns:** {overview['columns']}\n\n")
        
        f.write("## 2. AI-Assisted Evidence-Based Insights\n")
        f.write(f"{llm_insights}\n\n")
        
        f.write("## 3. Data Quality Overview\n")
        for check, result in quality.items():
            if isinstance(result, list):
                val_str = ", ".join(result) if result else "None detected"
                f.write(f"- **{check}:** {val_str}\n")
            elif isinstance(result, dict):
                f.write(f"- **{check}:** Handled across {len(result)} columns\n")
        f.write("\n")
        
        f.write("## 4. Visualizations\n")
        for plot in viz_info.get('generated_plots', []):
            f.write(f"![{plot}](plots/{plot})\n\n")
            
        if viz_info.get('skipped_plots'):
            f.write("### Skipped Visualizations\n")
            for plot_type, reason in viz_info['skipped_plots'].items():
                f.write(f"- **{plot_type}:** {reason}\n")

In [169]:
def custom_json_serializer(obj):
    """Converts NumPy and Pandas data types to standard Python types for JSON serialization."""
    if isinstance(obj, pd.Series):
        return obj.to_dict()
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict()
    if isinstance(obj, (np.integer, np.int64, np.int32)):
        return int(obj)
    if isinstance(obj, (np.floating, np.float64, np.float32)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, set):
        return list(obj)
    return str(obj)

In [170]:
def generate_profile(csv_path, output_base_dir="../output", use_llm=True, model_name="qwen2.5:0.5b"):
    """Master entry point for processing any CSV dataset."""
    dataset_filename = os.path.basename(csv_path)
    folder_name = os.path.splitext(dataset_filename)[0]
    
    output_dir = os.path.join(output_base_dir, folder_name)
    plots_dir = os.path.join(output_dir, "plots")
    os.makedirs(plots_dir, exist_ok=True)
    
    print(f"Reading dataset: {csv_path}")
    df = pd.read_csv(csv_path)
    
    roles = probable_role(df)
    overview = get_dataset_overview(df, dataset_filename, roles)
    quality = get_data_quality_profile(df, roles)
    stats = descriptive_statistics(df, roles)
    relationships = get_relationships_between_variables(df, roles)
    viz_info = generate_adaptive_visualizations(df, roles, plots_dir)
    
    analysis_summary = {
        "Dataset Overview": overview,
        "Data Quality": quality,
        "Descriptive Statistics": stats,
        "Relationships": relationships
    }
    
    with open(os.path.join(output_dir, 'analysis_summary.json'), 'w', encoding='utf-8') as f:
        json.dump(analysis_summary, f, default=custom_json_serializer, indent=4)
        
    col_profile = pd.DataFrame({
        'Column Name': list(overview['technical_type'].keys()),
        'Technical Type': list(overview['technical_type'].values()),
        'Probable Role': list(overview['probable_role'].values()),
        'Non-Missing Count': list(overview['non_missing_count'].values()),
        'Missing Percent': list(overview['missing_percent'].values()),
        'Unique Values': list(overview['unique_values'].values())
    })
    col_profile.to_csv(os.path.join(output_dir, 'column_profile.csv'), index=False)
    
    print("Generating LLM narrative...")
    clean_summary = json.loads(json.dumps(analysis_summary, default=custom_json_serializer))
    llm_insights = run_llm_insights(clean_summary, output_dir, use_llm, model_name)
    
    generate_markdown_report(dataset_filename, analysis_summary, viz_info, llm_insights, output_dir)
    
    print(f"Profiling complete for {dataset_filename}. Outputs saved to: {output_dir}")

In [171]:
# Dataset A (Provisional COVID-19 data)
generate_profile("../data/Provisional COVID-19 data.csv", use_llm=True, model_name="qwen2.5:0.5b")

Reading dataset: ../data/Provisional COVID-19 data.csv
Generating LLM narrative...
Profiling complete for Provisional COVID-19 data.csv. Outputs saved to: ../output\Provisional COVID-19 data


In [173]:
# Dataset B (NEH Grants 2000s data)
generate_profile("../data/NEH_Grants2000s.csv", use_llm=True, model_name="qwen2.5:0.5b")

Reading dataset: ../data/NEH_Grants2000s.csv
Generating LLM narrative...
Profiling complete for NEH_Grants2000s.csv. Outputs saved to: ../output\NEH_Grants2000s
